I will be building off of the chatbot defined in lab1 with my attempt at solving history conversations

### Import Libraries

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import random

In [ ]:
load_dotenv(override=True)

### Define LLM

In [ ]:
llm = ChatOpenAI(model = "gpt-4o-mini")

### Define State

In [ ]:
class State(BaseModel):
    query: Annotated[list, "User query"]
    output: Annotated[str, "Output of query"]
    messages: Annotated[list, add_messages]

### Define GraphBuilder

In [ ]:
graph_builder = StateGraph(State)

### Define Node

In [ ]:
def update_messages(state: State) -> State:
    new_message = f"User Query: {state.query}: LLM Output: {state.output}"
    return State(query=state.query, output=state.output, messages=state.messages + [new_message])

In [ ]:
def answer_query(state: State) -> State:
    response = llm.invoke(state.messages)
    return State(query=state.query, output=response.content, messages=[response])

In [ ]:
graph_builder.add_node("messages", update_messages)
graph_builder.add_node("answer", answer_query)

### Build Edges

In [ ]:
graph_builder.add_edge(START, "messages")
graph_builder.add_edge("messages", "answer")
graph_builder.add_edge("answer", END)

### Compile Graph

In [ ]:
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### Chat Interface

In [ ]:
def chat(user_input: str, history):
    state = State(
        query = [{"role" : "user", "content" : user_input}], 
        output = "", 
        messages=[]
    )
    result = graph.invoke(state)
    print(result)
    return result['output']

In [ ]:
gr.ChatInterface(chat, type="messages").launch()